# Lab 6 — Segmentation Summary

**Day 05 · Unsupervised Learning · Cisco AI/ML Training**

---

## Learning objectives

1. Assign K-Means **segments** (k=4) to NYSE symbols.
2. Summarize mean features per segment with `groupby`.
3. List representative ticker symbols per segment.
4. Translate clusters into **business-readable** segment labels.

> **Checkpoints:** sizes `{0:8, 1:9, 2:7, 3:1}` · segment means table · sample symbols per segment

**Companion script:** `../scripts/lab06_segmentation_summary.py`

## From cluster ID to business segment

Cluster labels (0, 1, 2, 3) are arbitrary — the value is the **profile** of each group:

| Segment profile | Typical signals |
|-----------------|----------------|
| High price / mega-cap | High `avg_close` |
| Mid price / diversified | Moderate price & volume |
| Higher volatility | Elevated `volatility` |
| Singleton | Unique profile (review manually) |

Use cases: portfolio grouping, risk bucketing, watchlist generation, sector-agnostic tilts.

---

## 1. Assign segments (K-Means k = 4)

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-05":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "nyse" / "nyse_stocks.csv").is_file():
            GH_ROOT = parent
            break

FEATURE_COLUMNS = ["avg_close", "volatility", "avg_volume", "avg_range"]

nyse = pd.read_csv(GH_ROOT / "data" / "nyse" / "nyse_stocks.csv", parse_dates=["date"])
nyse["range"] = nyse["high"] - nyse["low"]
features = (
    nyse.groupby("symbol")
    .agg(
        avg_close=("close", "mean"),
        volatility=("close", "std"),
        avg_volume=("volume", "mean"),
        avg_range=("range", "mean"),
    )
    .reset_index()
)
features["volatility"] = features["volatility"].fillna(0.0)

X_scaled = StandardScaler().fit_transform(features[FEATURE_COLUMNS])

k = 4
features = features.copy()
features["segment"] = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)

print("Lab 6 — Segmentation summary")
print(f"segments (k): {k}")

---

## 2. Segment sizes

In [ ]:
segment_sizes = features["segment"].value_counts().sort_index()
segment_sizes_dict = segment_sizes.to_dict()

print(f"symbols per segment: {segment_sizes_dict}")
display(segment_sizes.rename("count").to_frame())

---

## 3. Mean features per segment

In [ ]:
summary = (
    features.groupby("segment")[FEATURE_COLUMNS]
    .mean()
    .round(2)
    .reset_index()
)

print("segment means:")
display(summary)

---

## 4. Sample symbols per segment

In [ ]:
print("sample symbols per segment:")
sample_rows = []
for seg in sorted(features["segment"].unique()):
    symbols = features.loc[features["segment"] == seg, "symbol"].head(4).tolist()
    print(f"  segment {seg}: {symbols}")
    sample_rows.append({"segment": seg, "sample_symbols": ", ".join(symbols)})

display(pd.DataFrame(sample_rows))

---

## 5. Business-readable segment labels

In [ ]:
labels = {
    0: "high price / large-cap growth",
    1: "mid price / diversified large names",
    2: "lower price / higher volatility",
    3: "singleton — high price, low vol (PEP)",
}

summary_labeled = summary.copy()
summary_labeled["label"] = summary_labeled["segment"].map(labels)
display(summary_labeled)

---

## 6. Full symbol membership

In [ ]:
display(
    features[["symbol", "segment", "avg_close", "volatility"]]
    .sort_values(["segment", "symbol"])
    .round(2)
)

---

## 7. Segment size bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(
    x=segment_sizes.index.astype(str),
    y=segment_sizes.values,
    ax=ax,
    palette="Set2",
)
ax.set_xlabel("segment")
ax.set_ylabel("symbol count")
ax.set_title("NYSE K-Means segment sizes (k=4)")
plt.tight_layout()
plt.show()

---

## 8. Day 05 recap

In [ ]:
recap = pd.DataFrame({
    "lab": [
        "1 K-Means",
        "2 Elbow",
        "3 DBSCAN",
        "4 Metrics",
        "5 Multi-view",
        "6 Summary",
    ],
    "checkpoint": [
        "inertia ≈ 45.86",
        "best k = 3",
        "3 clusters, 11 noise",
        "silhouette ≈ 0.24",
        "multi_cluster_view.png",
        str(segment_sizes_dict),
    ],
})
display(recap)

---

## 9. Checkpoint summary

In [ ]:
assert k == 4
assert segment_sizes_dict == {0: 8, 1: 9, 2: 7, 3: 1}
assert summary.loc[summary["segment"] == 0, "avg_close"].iloc[0] > 200
seg0_symbols = features.loc[features["segment"] == 0, "symbol"].head(4).tolist()
assert seg0_symbols == ["CSCO", "DIS", "MSFT", "NFLX"]
print("✓ All checkpoint assertions passed")

---

## Reflection questions

1. Would you merge segment 3 (singleton PEP) into segment 0? What rule would you use?
2. One business use-case for each segment in a portfolio dashboard.
3. How does Day 06 fraud detection differ from today's unsupervised segmentation?

**Previous:** [Lab 5 — NYSE multi-cluster view](lab05_nyse_multi_cluster_view.ipynb)  
**Next:** [Day 06 — Anomaly Detection](../day-06/README.md)